# 03 · Pembagian Data dan Pembentukan Target
**Bab III Subbab 3.4 dan 3.5, Bab II Subbab 2.9.**

1. Pembagian *product-disjoint* 80/10/10 dan pemeriksaan kebocoran.
2. Subset bertingkat E1–E5.
3. Standardisasi fitur (statistik hanya dari data latih).
4. Model Negative Binomial dengan efek produk untuk mengukur hubungan usia dan helpful votes (RM3).
5. Target T1 (mentah) dan T3 (terkoreksi usia, *out-of-fold*), lalu tingkat relevansi 0–4.

**Keluaran:** `reviews_split.parquet`, `targets.parquet`, `split_report.json`, `age_model.json`, `tables/age_*.csv`

> **Google Colab.** Jalankan sekali per sesi sebelum sel setup:
> ```
> !git clone -b main https://github.com/Vito28/dl-paper.git
> %cd dl-paper/notebooks
> !pip install -q -r ../requirements.txt
> ```
> Sel setup otomatis me-mount Google Drive (izinkan akses saat diminta). **Semua data dan hasil, termasuk
> unduhan mentah, disimpan di `MyDrive/dl-paper`**, bukan di instance. Jika Drive gagal di-mount atau
> terputus, notebook berhenti dengan error. Jalankan sel terakhir (`finish()`) agar semua tulisan tersinkron.

In [ ]:
# Setup: path proyek dan konfigurasi bersama (ubah konfigurasi di notebooks/settings.py)
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "notebooks")]
from settings import cfg, SMOKE, finish
import numpy as np, pandas as pd
pd.set_option("display.width", 160)
print("kategori:", cfg.category, "| output:", cfg.out_dir, "| SMOKE:", SMOKE)

In [ ]:
from hr import split, targets, plots
import matplotlib.pyplot as plt
df = pd.read_parquet(cfg.path("reviews_clean.parquet"))
print(df.shape)

## 1. Pembagian product-disjoint

In [ ]:
df = split.split_products(df, cfg)
df, leak = split.leakage_checks(df)
df, cap = split.cap_eval(df, cfg)
df = split.make_scales(df, cfg)
std = split.fit_standardizer(df)
df = split.apply_standardizer(df, std)
split_report = {"leakage": leak, "eval_cap": cap, "standardizer": std}
cfg.path("split_report.json").write_text(json.dumps(split_report, indent=2, default=str))
split_report["leakage"]

In [ ]:
scale_tab = split.scale_report(df, cfg)
scale_tab.to_csv(cfg.path("tables", "split_dan_skala.csv"), index=False)
scale_tab

In [ ]:
# Pemeriksaan: subset bertingkat dan tidak ada produk validasi/uji di data latih
names = list(cfg.scales)
for a, b in zip(names, names[1:]):
    assert not (df[f"in_{a}"] & ~df[f"in_{b}"]).any(), f"{a} bukan bagian dari {b}"
assert set(df.loc[df.split != "train", "parent_asin"]).isdisjoint(df.loc[df.split == "train", "parent_asin"])
print("OK")

In [ ]:
df.to_parquet(cfg.path("reviews_split.parquet"), index=False)

## 2. Model hitung usia (persamaan 2.16–2.17)
Diestimasi pada data latih skala terbesar. Bentuk fungsi usia (linear atau spline) dipilih dari *log-likelihood* data validasi terbesar (bukan *deviance*, karena setiap basis punya dispersi κ sendiri).
Pada data validasi, efek produk baru diestimasi ulang sementara fungsi usia dibekukan, karena efek produk hanya parameter pengganggu.

In [ ]:
largest = list(cfg.scales)[-1]
tr = df[(df.split == "train") & df[f"in_{largest}"]].reset_index(drop=True)
va = df[df.split == "val"].reset_index(drop=True)
te = df[df.split == "test"].reset_index(drop=True)
best_kind, basis_tab = targets.select_basis(tr, va, cfg.age_basis_grid)
basis_tab.to_csv(cfg.path("tables", "age_pemilihan_basis.csv"), index=False)
print("basis terpilih:", best_kind)
basis_tab

In [ ]:
# Kontribusi usia = uji LR dan loglik_gain_share (pseudo-R2 McFadden terhadap model tanpa usia).
# deviance_full dan deviance_noage memakai kappa masing-masing, jadi tidak dibandingkan langsung.
full = targets.fit_age_model(tr, basis_kind=best_kind)
noage = targets.fit_age_model(tr, use_age=False)
summ = targets.age_effect_summary(full, noage)
summ["basis"] = str(best_kind)
summ["beta"] = dict(zip(targets.COVARIATES, full.beta.round(4).tolist()))
cfg.path("age_model.json").write_text(json.dumps(summ, indent=2, default=str))
pd.Series(summ)

### Kurva usia dengan IK 95% (bootstrap per produk)
Bootstrap mengambil ulang produk dari seluruh data latih yang sama dengan model di atas, dengan knot spline tetap.
Nilai 2,0 berarti rata-rata suara yang diharapkan dua kali lipat dibanding ulasan berusia 30 hari, setelah produk dan kovariat dikendalikan. Ini asosiasi, bukan sebab-akibat.

In [ ]:
ages = np.unique(np.round(np.logspace(0, np.log10(tr["age_days"].max()), 40)))
boots = targets.bootstrap_age_curve(tr, full, ages, n_boot=cfg.age_bootstrap, seed=cfg.seed)
curve_tab = targets.age_curve_table(full, boots, ages)
curve_tab.to_csv(cfg.path("tables", "age_kurva.csv"), index=False)
ax = plots.age_curve(curve_tab["age_days"], curve_tab["ratio"], curve_tab["lo"], curve_tab["hi"])
plots.save(ax.figure, cfg.path("figures", "age_kurva.png"))

## 3. Target T1 dan T3
f̂ untuk data latih dihitung *out-of-fold* (fold dikelompokkan per produk). Data validasi dan uji memakai model dari seluruh data latih.

In [ ]:
f_oof = targets.oof_age_component(tr, best_kind, cfg.n_folds_age, cfg.seed)
f_bar = float(f_oof.mean())
f_all = {"train": f_oof, "val": full.f(va["log_age"].to_numpy()), "test": full.f(te["log_age"].to_numpy())}
parts = []
for name, d in (("train", tr), ("val", va), ("test", te)):
    parts.append(pd.DataFrame({"rid": d["rid"].to_numpy(), "f_hat": f_all[name],
                               "y_t1": targets.target_t1(d["helpful_vote"]),
                               "y_t3": targets.target_t3(d["helpful_vote"], f_all[name], f_bar)}))
tg = pd.concat(parts, ignore_index=True)
is_tr = tg["rid"].isin(tr["rid"])
q1 = targets.fit_grade_thresholds(tg.loc[is_tr, "y_t1"])
q3 = targets.fit_grade_thresholds(tg.loc[is_tr, "y_t1"], tg.loc[is_tr, "y_t3"])
tg["grade_t1"] = targets.grades(tg["y_t1"], q1)
tg["grade_t3"] = targets.grades(tg["y_t3"], q3)
assert np.isfinite(tg[["y_t1", "y_t3"]].to_numpy()).all(), "target T1/T3 berisi NaN/inf"
assert len(tg) == tg["rid"].nunique() == len(tr) + len(va) + len(te)
tg.to_parquet(cfg.path("targets.parquet"), index=False)
thr = {"q_t1": q1.tolist(), "q_t3": q3.tolist(), "f_bar": f_bar, "map_relevant_grade": cfg.map_relevant_grade}
cfg.path("thresholds.json").write_text(json.dumps(thr, indent=2))
thr

In [ ]:
# Sebaran tingkat relevansi per partisi (dilaporkan di Bab IV)
sp = df.set_index("rid")["split"]
tg["split"] = tg["rid"].map(sp)
grade_tab = pd.concat({p: tg.groupby("split")[f"grade_{p}"].value_counts(normalize=True).unstack().round(3)
                       for p in ("t1", "t3")}, axis=0)
grade_tab.to_csv(cfg.path("tables", "sebaran_tingkat_relevansi.csv"))
grade_tab

In [ ]:
# Batas kelompok usia (baru, menengah, lama) dari data latih, dipakai di notebook 07
edges = np.quantile(tr["age_days"], [1/3, 2/3]).tolist()
cfg.path("age_group_edges.json").write_text(json.dumps(edges))
edges

## Simpan ke Google Drive
Memastikan semua hasil notebook ini sudah tersinkron ke Drive sebelum runtime ditutup.

In [ ]:
finish()